In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from PIL import Image
from typing import Union

import math
import PIL.Image
import torch
import torch.nn.functional as F
from torch import nn
from einops import rearrange
import PIL
from torchvision.transforms.v2 import (
    Compose,
    Resize,
    InterpolationMode,
    ToImage,
    ToDtype,
    Normalize,
)
from transformers.utils import is_flash_attn_2_available

try:
    if is_flash_attn_2_available():
        from flash_attn.modules.mha import FlashSelfAttention
    else:
        FlashSelfAttention = None
except ImportError:
    FlashSelfAttention = None


class Attention(nn.Module):

    def __init__(self, dim, num_heads=16, use_flash_attn=False):
        super().__init__()
        assert dim % num_heads == 0, "dim should be divisible by num_heads"
        
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        if use_flash_attn and FlashSelfAttention is not None:
            self.flash_attn = FlashSelfAttention()
        else:
            self.flash_attn = None

        torch.nn.init.kaiming_normal_(
            self.qkv.weight, mode="fan_in", nonlinearity="relu"
        )
        torch.nn.init.kaiming_normal_(
            self.proj.weight, mode="fan_in", nonlinearity="relu"
        )


    def custom_scaled_dot_product_attention(self, query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
        # Get sequence lengths
        L, S = query.size(-2), key.size(-2)
        
        # Scale factor based on the size of the query dimension
        scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
        
        # Initialize attention bias (for masking)
        attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
        
        # Handle causal masking
        if is_causal:
            assert attn_mask is None, "Causal masking and attention mask should not be applied together."
            temp_mask = torch.ones(L, S, dtype=torch.bool, device=query.device).tril()
            attn_bias.masked_fill_(~temp_mask, float("-inf"))
        
        # Handle attention mask (if provided)
        if attn_mask is not None:
            if attn_mask.dtype == torch.bool:
                attn_bias.masked_fill_(~attn_mask, float("-inf"))
            else:
                attn_bias += attn_mask
        
        # Compute attention scores (scaled dot product)
        attn_weights = torch.matmul(query, key.transpose(-2, -1)) * scale_factor
        
        # Add attention bias (e.g., for masking)
        attn_weights += attn_bias
        
        # Apply softmax to get attention probabilities
        attn_weights = torch.softmax(attn_weights, dim=-1)
        
        # Apply dropout (if training)
        if dropout_p > 0.0:
            attn_weights = F.dropout(attn_weights, p=dropout_p, training=True)
        
        # Compute the final output by applying attention weights to values
        output = torch.matmul(attn_weights, value)
        return output

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.flash_attn is not None:
            qkv = self.qkv(x)
            qkv = rearrange(
                qkv, "... (three h d) -> ... three h d", three=3, h=self.num_heads
            )
            attn_output = self.flash_attn(qkv)
            output = rearrange(attn_output, "... h d -> ... (h d)")
            output = self.proj(output)
            return output
        else:
            B, N, C = x.shape
            qkv = (
                self.qkv(x)
                .reshape(B, N, 3, self.num_heads, self.head_dim)
                .permute(2, 0, 3, 1, 4)
            )
            q, k, v = qkv.unbind(0)

            x = F.scaled_dot_product_attention(q, k, v)

            x = x.transpose(1, 2).reshape(B, N, C)
            x = self.proj(x)
            return x


class VitBlock(nn.Module):

    def __init__(self, embed_dim, use_flash_attn=False):
        super().__init__()
        self.attn = Attention(embed_dim, use_flash_attn=use_flash_attn)
        self.mlp = MLP(embed_dim, 4304)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):

    def __init__(self, use_flash_attn=False):
        super().__init__()

        embed_len = 729
        embed_dim = 1152

        self.patch_embed = LinearPatchEmbedding()
        self.pos_embed = nn.Parameter(torch.randn(1, embed_len, embed_dim) * 0.02)
        self.blocks = nn.Sequential(
            *[VitBlock(embed_dim, use_flash_attn=use_flash_attn) for _ in range(27)]
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


class EncoderWrapper(nn.Module):

    def __init__(self, use_flash_attn=False):
        super().__init__()
        self.model = nn.ModuleDict({"visual": VisionTransformer(use_flash_attn)})

    def forward(self, x):
        return self.model["visual"](x)


class LinearPatchEmbedding(nn.Module):

    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(588, 1152)

    def forward(self, x):
        b, c, hp1, wp2 = x.shape
        p1, p2 = 14, 14
        h, w = hp1 // p1, wp2 // p2
        x = x.reshape(b, c, h, p1, w, p2)
        x = x.permute(0, 2, 4, 1, 3, 5)
        x = x.reshape(b, h * w, c * p1 * p2)
        return self.linear(x)


class MLP(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: int = None,
        out_features: int = None,
    ) -> None:
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU(approximate="tanh")
        self.fc2 = nn.Linear(hidden_features, out_features)

        torch.nn.init.kaiming_normal_(
            self.fc1.weight, mode="fan_in", nonlinearity="relu"
        )
        torch.nn.init.kaiming_normal_(
            self.fc2.weight, mode="fan_in", nonlinearity="relu"
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x


class VisionProjection(nn.Module):
    def __init__(self):
        super().__init__()

        image_embedding_dim = 1152
        model_dim = 2048
        hidden_dim = model_dim * 4

        self.mlp = MLP(image_embedding_dim * 2, hidden_dim, model_dim)

    @property
    def device(self):
        return self.mlp.fc1.weight.device

    def forward(self, x):
        return self.mlp(x)


def create_patches(image, patch_size=(378, 378)):
    assert image.dim() == 3, "Image must be in CHW format"

    _, height, width = image.shape  # Channels, Height, Width
    patch_height, patch_width = patch_size

    if height == patch_height and width == patch_width:
        return []

    # Iterate over the image and create patches
    patches = []
    for i in range(0, height, patch_height):
        row_patches = []
        for j in range(0, width, patch_width):
            patch = image[:, i : i + patch_height, j : j + patch_width]
            row_patches.append(patch)
        patches.append(torch.stack(row_patches))
    return patches


class VisionEncoder(nn.Module):

    def __init__(self, use_flash_attn=False):
        super().__init__()

        self.encoder = EncoderWrapper(use_flash_attn)
        self.projection = VisionProjection()
        self.supported_sizes = [(378, 378), (378, 756), (756, 378), (756, 756)]

    @property
    def device(self):
        return self.projection.mlp.fc1.weight.device

    @property
    def dtype(self):
        return self.projection.mlp.fc1.weight.dtype

    def preprocess(self, image: PIL.Image.Image):
        width, height = image.size
        max_dim = max(width, height)
        if max_dim < 512:
            im_size = (378, 378)
        else:
            aspect_ratio = width / height
            im_size = min(
                self.supported_sizes,
                key=lambda size: (
                    abs((size[1] / size[0]) - aspect_ratio),
                    abs(size[0] - width) + abs(size[1] - height),
                ),
            )

        return Compose(
            [
                Resize(size=im_size, interpolation=InterpolationMode.BICUBIC),
                ToImage(),
                ToDtype(torch.float32, scale=True),
                Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            ]
        )(image)

    def forward(
        self, images: Union[PIL.Image.Image, list[PIL.Image.Image], torch.Tensor]
    ) -> torch.Tensor:
        im_list = None
        if isinstance(images, torch.Tensor):
            # Input must have dimensions (B, C, H, W)
            assert (
                len(images.shape) == 4
            ), "Tensor input must have dimensions (B, C, H, W)"
            im_list = list(images)
        elif isinstance(images, PIL.Image.Image):
            im_list = [images]
        elif isinstance(images, list):
            im_list = images
        else:
            raise ValueError(
                "Input must be a PIL image, list of PIL images, or a tensor"
            )

        # Preprocess unless the images are already tensors (indicating that
        # they have already been preprocessed)
        if not isinstance(im_list[0], torch.Tensor):
            im_list = [self.preprocess(im.convert("RGB")) for im in im_list]

        patches = [create_patches(im) for im in im_list]
        flat_patches = [patch for image_patches in patches for patch in image_patches]

        # Images may be variable size, and need to be resized to a common size after
        # creating patches.
        resized_images = [
            F.interpolate(im.unsqueeze(0), size=(378, 378), mode="bilinear")
            for im in im_list
        ]

        combined_images = torch.cat([*resized_images, *flat_patches], dim=0)
        combined_images = combined_images.to(self.device, dtype=self.dtype)


        combined_features = self.encoder(combined_images)


        full_img_features = combined_features[: len(im_list)]
        patch_features = (
            combined_features[len(im_list) :].transpose(1, 2).view(-1, 1152, 27, 27)
        )

        # Reshape patch features back to their original structure
        reshaped_patch_features = []
        patch_idx = 0
        for i, patch_set in enumerate(patches):
            if len(patch_set) == 0:
                print("Inside IF")
                reshaped_patch_features.append(
                    full_img_features[i].transpose(0, 1).view(1152, 27, 27)
                )
            else:
                sample_features = []
                for row_patches in patch_set:
                    row_len = len(row_patches)
                    row_features = patch_features[
                        patch_idx : patch_idx + row_len
                    ]  # row_len, T, C
                    row_features = torch.cat(
                        list(row_features), dim=2
                    )  # T, C * row_len
                    patch_idx += row_len
                    sample_features.append(row_features)
                sample_features = torch.cat(sample_features, dim=1)
                sample_features = F.interpolate(
                    sample_features.unsqueeze(0), size=(27, 27), mode="bilinear"
                ).squeeze(0)
                reshaped_patch_features.append(sample_features)

        transposed_features = full_img_features[0].transpose(0, 1)
        reshaped_features = transposed_features.reshape(1152, 27, 27)

        print("transposed_features", transposed_features, transposed_features.shape)
        print("reshaped_features:", reshaped_features, reshaped_features.shape)
        print("full_img_features:", full_img_features, full_img_features.shape)
        print("reshaped_patch_features:", reshaped_patch_features, len(reshaped_patch_features), reshaped_patch_features[0].shape)

        reshaped_patch_features = (
            torch.stack(reshaped_patch_features).view(-1, 1152, 729).transpose(1, 2)
        )

        final_features = torch.cat([full_img_features, reshaped_patch_features], dim=2)

        return self.projection(final_features)

In [2]:
image = Image.open('download.jpeg')

In [3]:
x = VisionEncoder(image)
x.load_state_dict(torch.load("model_weights.pth"))
# Calculate the total number of parameters
total_params = sum(p.numel() for p in x.parameters())

print(f"Total number of parameters: {total_params}")

/tmp/ipykernel_20298/3745341823.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x.load_state_dict(torch.load("model_weights.pth"))


Total number of parameters: 448649072


In [4]:
y = x.forward(image)

Inside IF
transposed_features tensor([[ 0.3574, -0.0712, -0.0544,  ..., -0.1882,  0.1523,  0.2065],
        [-0.0295, -0.1158, -0.1217,  ...,  0.2571,  0.3018,  0.3379],
        [ 0.5589,  0.0639,  0.0707,  ...,  0.3509,  0.1395,  0.1097],
        ...,
        [ 0.0705, -0.0622, -0.0494,  ...,  0.0144, -0.1067,  0.0843],
        [-0.3611,  0.1577,  0.1573,  ..., -0.3240,  0.1965,  0.1908],
        [ 0.0805,  0.0021, -0.0088,  ...,  0.0150, -0.3924, -0.3402]],
       grad_fn=<TransposeBackward0>) torch.Size([1152, 729])
reshaped_features: tensor([[[ 3.5737e-01, -7.1188e-02, -5.4447e-02,  ..., -4.4229e-01,
          -2.2349e-01, -5.2457e-02],
         [-4.0758e-01,  1.8471e-01, -1.9905e-01,  ...,  3.9131e-02,
          -5.0238e-01, -2.9365e-03],
         [-3.3587e-01, -4.7483e-02,  2.5824e-01,  ..., -3.9282e-01,
           7.1325e-02, -1.7226e-01],
         ...,
         [-4.4596e-01, -3.5458e-03, -7.0351e-03,  ..., -3.3549e-01,
          -3.9132e-01, -2.3992e-01],
         [ 2.7285e-01,

In [5]:
# Iterate through the state_dict to get the layer names and their corresponding shapes
for name, param in x.state_dict().items():
    print(f"Layer: {name}, Shape: {param.shape}")

Layer: encoder.model.visual.pos_embed, Shape: torch.Size([1, 729, 1152])
Layer: encoder.model.visual.patch_embed.linear.weight, Shape: torch.Size([1152, 588])
Layer: encoder.model.visual.patch_embed.linear.bias, Shape: torch.Size([1152])
Layer: encoder.model.visual.blocks.0.attn.qkv.weight, Shape: torch.Size([3456, 1152])
Layer: encoder.model.visual.blocks.0.attn.qkv.bias, Shape: torch.Size([3456])
Layer: encoder.model.visual.blocks.0.attn.proj.weight, Shape: torch.Size([1152, 1152])
Layer: encoder.model.visual.blocks.0.attn.proj.bias, Shape: torch.Size([1152])
Layer: encoder.model.visual.blocks.0.mlp.fc1.weight, Shape: torch.Size([4304, 1152])
Layer: encoder.model.visual.blocks.0.mlp.fc1.bias, Shape: torch.Size([4304])
Layer: encoder.model.visual.blocks.0.mlp.fc2.weight, Shape: torch.Size([1152, 4304])
Layer: encoder.model.visual.blocks.0.mlp.fc2.bias, Shape: torch.Size([1152])
Layer: encoder.model.visual.blocks.0.norm1.weight, Shape: torch.Size([1152])
Layer: encoder.model.visual.blo